# CwH Stage 2b v2 — Stage 2 survivors under the SELECTION null; timing deferred (CONTROLS, not trials)

**SUPERSEDES Stage 2b v1.** v1's decisive null 1b (same ticker, same calendar month) was shown to be PATH-BIASED: on a
martingale world (no edge possible) it rewarded entries after declines and put breakouts at the 0th percentile, because
a same-month draw is conditioned on the path that defines the signal. v1's tiers are void; these replace them.

**STANDING RULE:** every null must first pass the 16-SEED MARTINGALE PROTOCOL — >= 16 independent pure-random-walk worlds
(no planted patterns; bars from an intrabar random path, so High/Low are levels the price actually traded through), a
config subset spanning every entry family; per null: pooled pass rate <= 10%, each family's mean percentile across seeds
within 40-60 with |t| < 2.5, no family pass rate > 15%. A failing null STOPS the stage; only then planted-alpha worlds,
then real data. The SELECTION null used here passed it; both timing-null versions failed it.

Same lockbox as Stage 2: every series cut at 2024-12-31 at load; IS entries 2011-01-01 .. 2023-12-29. Same simulator,
realistic fills, 0.15% price costs, 10% max stop, per-ticker non-overlap. Nothing here is a trial.

**THIS RUN: SELECTION null only — TIMING DEFERRED.** The order-matched timing null failed the martingale protocol
(path-biased) and the unconditioned timing null failed it too (centred but over-dispersed: ~27% of percentiles in the
5% tails on pure noise). No timing null is computed here; a paired-difference timing test is being validated separately.

**SELECTION null** (validated: 16-seed martingale protocol PASSED — pooled pass 6.2%, family means 51-58, |t| <= 2.04):
same anchor date, a random OTHER ticker with v2b true that day, same order type on that date (next-open -> next open;
buy-stop -> that ticker's prior-bar high, fills next bar at max(open, level), redraw if not hit); stops 7% exact /
handle = the matched trade's own distance; exit levels = matched % distances; null entries <= the purge date;
non-overlap per draw; 200 draws.
**Supporting only:** FIXED-HORIZON excess vs the EW universe and SPY at **60 and 120 bars** (close of the entry/fill bar ->
close h bars later, same close-to-close window for the benchmark, independent of each trade's stop / exit; year-block 90%
CI). Both horizons PASSED the 16-seed martingale protocol (pooled pass 7.1% / 7.6%, family means 45-57, |t| <= 1.4).
**20 and 250 bars were DROPPED on synthetic-validity grounds BEFORE any real-data run:** at 20 bars the breakout family's
mean percentile was 37.2 (t -2.17) and at 250 bars the days family's was 60.9 (t +2.49) — outside the 40-60 band on pure
noise. (An earlier open-start window failed outright: it included the intrabar move that triggered a buy-stop / limit fill.) · 4b generic 52-week breakout draws (other tickers, same month) · 4a full generic-breakout
strategies · IS buy-and-hold per bar. v1's EW row measured the benchmark over each trade's OWN window, whose end is set
by the stock's own stop (which fires mostly when the shared market falls) -> biased upward on pure noise; replaced.

**PRE-REGISTERED READING RULE — selection-only run (fixed here, before any result):**
- **TIER S "selection passes (timing untested)":** SELECTION >= 95th pct on PF_cost AND avg % return.
- **TIER 3 "no detectable alpha":** SELECTION fails.
4b and fixed-horizon excess are reported beside the tier and do not change it. v1 tiers stay SUPERSEDED.
**The lockbox stays sealed regardless of tier.**


In [ ]:
# ===== Cell 0 — Stage 2 config (verbatim) + Stage 2b v2 constants =====
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, glob, time, json, hashlib, os, itertools, math
from pathlib import Path
from numba import njit, prange
import warnings; warnings.filterwarnings('ignore')

DATA_DIR = Path('/content/drive/MyDrive/Bukowski')
HIST_DIR = DATA_DIR/'results'/'history_15yr'          # READ-ONLY: PIT_C outputs (A1 parity)
S1_DIR   = DATA_DIR/'results'/'sweep_PIT_C'           # the append-only trial log lives here
REF_DIR  = DATA_DIR/'reference'
OUT_DIR  = DATA_DIR/'results'/'stage2_PIT_C'; CACHE = OUT_DIR/'cache'
CACHE.mkdir(parents=True, exist_ok=True)
RSP_PATH = DATA_DIR/'RSP.csv' if (DATA_DIR/'RSP.csv').exists() else REF_DIR/'RSP.csv'

# ---- LOCKBOX ----
CUT_DATE   = pd.Timestamp('2024-12-31')     # every series cut at load
PURGE_DATE = pd.Timestamp('2023-12-29')     # IS entries <= this (covers the 250-bar time stop)
IS_START   = pd.Timestamp('2011-01-01')
IS_YEARS   = list(range(2011, 2024))         # 13 CPCV groups
N_MONTHS   = 12 * len(IS_YEARS)

# ---- PIT_C machinery (fixed, unsourced) ----
ATR_PERIOD, STOCK_FAST_SMA, STOCK_SLOW_SMA = 14, 50, 200
PIV, RIM_TOL, ROUND_MIN, ROUND_BAND, ROUND_SPAN = 5, 0.15, 3, 0.05, 5
HANDLE_WINDOW, CONFIRM_WINDOW_BARS = 30, 15
MIN_STOP_ATR, MAX_STOP_PCT = 1.0, 0.10
ENUM_DUR, ENUM_DEPTH = (30, 325), (0.12, 0.50)    # widest bounds over all variants
TIME_STOP_DAYS = 120                              # PIT_C (parity only)

VARIANTS = {
    'A1': dict(dur=(30, 130),  depth=(0.12, 0.50), handle='pitc'),
    'A2': dict(dur=(35, 325),  depth=(0.12, 0.33), handle='oneil', handle_max_drop=0.15, prior_up=0.30, prior_lb=252),
    'A4': dict(dur=(60, 130),  depth=(0.12, 0.50), handle='pitc'),
    'A5': dict(dur=(30, 130),  depth=(0.12, 0.33), handle='pitc'),
    'A6': dict(dur=(30, 130),  depth=(0.12, 0.50), handle='pitc_upper'),
}
A6_MIN_DIFF = 0.10

# ---- Tier B dimensions ----
ENTRIES = ['next_open', 'buy_stop']
STOPS   = ['handle', 'pct7', 'pct8', 'atr2', 'atr3']
TARGETS = ['t05', 't075', 't10', 't15', 'p15', 'p20', 'p25', 'none']
TRAILS  = ['none', 'sma10', 'sma21', 'sma50', 'chand3']
TIMES   = [60, 120, 250]
GATES   = ['off', 'on']
TARGET_NBRS = {'t05': ['t075'], 't075': ['t05', 't10'], 't10': ['t075', 't15'], 't15': ['t10', 'none'],
               'p15': ['p20'], 'p20': ['p15', 'p25'], 'p25': ['p20', 'none'], 'none': ['t15', 'p25']}
TRAIL_NBRS  = {'sma10': ['sma21'], 'sma21': ['sma10', 'sma50'], 'sma50': ['sma21', 'none'],
               'none': ['sma50', 'chand3'], 'chand3': ['none']}
TIME_NBRS   = {60: [120], 120: [60, 250], 250: [120]}
CONTROL = dict(variant='A1', entry='next_open', stop='handle', target='t05', trail='none', time=120, gate='off')
COST_PCT, CHAND_MULT = 0.0015, 3.0

GATE = dict(min_pf_cost=1.30, min_ci_lo=1.0, ci_level=0.90, boot_reps=2000, min_n=300,
            min_years_beat=9, n_is_years=13, max_year_share=0.25, plateau_median_pf=1.20,
            max_pbo=0.20, min_dsr=0.95)
PBO_CFG  = dict(groups='IS years 2011-2023', n_test=6, metric='PF_cost', min_n_train=100)
EFFN_CFG = dict(method='hierarchical agglomerative', linkage='average', distance='1 - Pearson rho, monthly cost-adjusted R',
                threshold=0.50, population='Stage 1 (212) + Stage 2', zero_variance='own cluster')
DSR_CFG  = dict(series='monthly cost-adjusted R, 156 IS months, empty = 0', sr='mean/std ddof=1',
                kurtosis='Pearson', var_sr='across Stage 2 configs', n_gate='effective', n_sensitivity='raw')
SWEEP_ID = 'PIT_C_stage2_v1'
BUILT_FROM_COMMIT = '1105333e2c55fdc5bc28bf57f8390621ac78428b'
STAGE1_TRIALS = 212

print('LOCKBOX: cut', CUT_DATE.date(), '| IS entries', IS_START.date(), '..', PURGE_DATE.date(), f'| {len(IS_YEARS)} IS years')
print('GATES (fixed):', json.dumps(GATE)); print('PBO:', PBO_CFG); print('EFFECTIVE N:', EFFN_CFG); print('DSR:', DSR_CFG)
print('variants:', list(VARIANTS), '| Tier B per variant:', len(ENTRIES)*len(STOPS)*len(TARGETS)*len(TRAILS)*len(TIMES)*len(GATES))

OUT2B_V1 = DATA_DIR/'results'/'stage2b_PIT_C'
OUT2B = DATA_DIR/'results'/'stage2b_v2_PIT_C'; OUT2B.mkdir(parents=True, exist_ok=True)
N_DRAWS, BOOT_REPS, CI_LEVEL, TIER_PCT = 200, 5000, 0.90, 95.0
BREAKOUT_LOOKBACKS = [252, 50]
GATE3 = dict(timing_window=21, n_draws=N_DRAWS)
print('STAGE 2b v2 — SELECTION null only (timing deferred), controls not trials | draws', N_DRAWS, '| tier percentile', TIER_PCT)


In [ ]:
# ===== Cell 1 — Load prices CUT AT LOAD, trailing indicators, regime series  (~5-8 min) =====
def compute_atr(df, period=14):
    h,l,c = df['High'],df['Low'],df['Close']; pc=c.shift(1)
    tr = pd.concat([(h-l),(h-pc).abs(),(l-pc).abs()],axis=1).max(axis=1)
    return tr.rolling(period).mean()

def load_px(path):
    """Read OHLC(+V), CUT at CUT_DATE before anything else touches it. Returns (cut_df, n_full_rows).
    n_full_rows (a row COUNT only) keeps the >=260-bar universe rule identical to PIT_C."""
    cols = pd.read_csv(path, nrows=0).columns
    use = ['Date','Open','High','Low','Close'] + (['Volume'] if 'Volume' in cols else [])
    d = pd.read_csv(path, usecols=use); d['Date'] = pd.to_datetime(d['Date'])
    d = d.sort_values('Date').drop_duplicates('Date').reset_index(drop=True)
    n_full = len(d)
    d = d[d['Date'] <= CUT_DATE].reset_index(drop=True)                  # LOCKBOX
    if 'Volume' not in d.columns: d['Volume'] = np.nan
    return d, n_full

def to_arrays(d):
    d = d.copy()
    d['ATR']  = compute_atr(d, ATR_PERIOD)
    d['S50']  = d['Close'].rolling(STOCK_FAST_SMA).mean()
    d['S200'] = d['Close'].rolling(STOCK_SLOW_SMA).mean()
    return dict(Date=d['Date'].values, Open=d['Open'].to_numpy(float), High=d['High'].to_numpy(float),
                Low=d['Low'].to_numpy(float), Close=d['Close'].to_numpy(float), Volume=d['Volume'].to_numpy(float),
                ATR=d['ATR'].to_numpy(float), S50=d['S50'].to_numpy(float), S200=d['S200'].to_numpy(float))

def read_is_only(path, cols):
    """Read only rows with entry_date <= PURGE_DATE (dates first, every other row skipped), so no
    post-purge row is ever loaded. Row order of the file is preserved."""
    dts = pd.to_datetime(pd.read_csv(path, usecols=['entry_date'])['entry_date'])
    keep = set((dts.index[dts <= PURGE_DATE] + 1).tolist())          # +1: header is line 0
    date_cols = [c for c in cols if c.endswith('_date')]
    return pd.read_csv(path, usecols=cols, parse_dates=date_cols,
                       skiprows=lambda i: i > 0 and i not in keep)[cols]

nan = np.nan
def _idx_le(dates, d):
    """last index with date <= d (-1 if none)"""
    return int(np.searchsorted(dates, np.datetime64(d), side='right')) - 1

def mkt_features(d, spy, rsp):
    out = dict(spy_gt_200=nan, spy_50_gt_200=nan, rsp_spy_20d=nan)
    k = _idx_le(spy['Date'], d); c = spy['Close']
    if k >= 199:
        s200 = c[k-199:k+1].mean()
        out['spy_gt_200'] = float(c[k] > s200)
        out['spy_50_gt_200'] = float(c[k-49:k+1].mean() > s200)
    r = _idx_le(rsp['Date'], d)
    if k >= 20 and r >= 20:
        out['rsp_spy_20d'] = (rsp['Close'][r]/rsp['Close'][r-20] - 1) - (c[k]/c[k-20] - 1)
    return out


def breadth_window(W):
    """THE breadth definition — the ONLY implementation, called by both the fast path and the cut-off
    recompute on bitwise-identical inputs. W = the last <=50 rows (dates <= d) x tickers, last row = d.
    Eligible on d: a close on d AND 50 non-missing closes in the 50 rows ending at d (only bars <= d).
    Returns (breadth, n_eligible). One helper on one array layout -> identical floats; pandas'
    rolling mean returns a flat window EXACTLY while a direct mean can land 1 ulp low, which flipped
    close > SMA50 on stale/flat stretches (the 2011-12 mismatches)."""
    W = np.ascontiguousarray(W, dtype=float)
    if W.shape[0] < 50: return nan, 0
    ok = ~np.isnan(W).any(axis=0)
    sma = W.sum(axis=0) / 50.0
    n_el = int(ok.sum())
    return (float((W[-1] > sma)[ok].sum()) / n_el if n_el else nan), n_el

def build_panels(src, cal):
    """Universe panels on the SPY calendar. Row d depends only on rows <= d."""
    close = pd.DataFrame({tk: pd.Series(p['Close'], index=pd.DatetimeIndex(p['Date'])) for tk, p in src.items()})
    close = close.reindex(pd.DatetimeIndex(cal))
    A = np.ascontiguousarray(close.to_numpy(float))
    bn = [breadth_window(A[max(0, t-49):t+1]) for t in range(len(A))]
    breadth = pd.Series([b for b, _ in bn], index=close.index)
    n_elig = pd.Series([n for _, n in bn], index=close.index)
    rs6  = (close / close.shift(126) - 1).rank(axis=1, pct=True)
    rs12 = (close / close.shift(252) - 1).rank(axis=1, pct=True)
    return dict(close=close, breadth=breadth, n_elig=n_elig, rs6=rs6, rs12=rs12)


def add_trail_arrays(p):
    c = pd.Series(p['Close'])
    for k in (10, 21, 50): p[f'SMA{k}'] = c.rolling(k).mean().to_numpy(float)
    return p

def regime_series(panels, spy, rsp, cal):
    """Daily market regime on the SPY calendar: breadth_50 (shared Stage 1 breadth_window) and
    rsp_spy_20d (Stage 1 mkt_features), both from bars <= the date only."""
    rows = [mkt_features(pd.Timestamp(d), spy, rsp)['rsp_spy_20d'] for d in cal]
    return pd.DataFrame({'breadth_50': panels['breadth'].to_numpy(float), 'rsp_spy_20d': rows},
                        index=pd.DatetimeIndex(cal))
# ---- run ----
files = sorted(f for f in glob.glob(f'{DATA_DIR}/*.csv') if Path(f).stem.upper() != 'SPY')
price_db = {}; N_FULL = {}; PANEL_SRC = {}; t0 = time.time()
for i, f in enumerate(files):
    tk = Path(f).stem.upper()
    try:
        d, n_full = load_px(f)
        if len(d) == 0: continue
        PANEL_SRC[tk] = dict(Date=d['Date'].values, Close=d['Close'].to_numpy(float))
        if n_full < 260: continue
        price_db[tk] = add_trail_arrays(to_arrays(d)); N_FULL[tk] = n_full
    except Exception:
        pass
    if (i+1) % 400 == 0: print(f'  loaded {len(price_db)} / {i+1} ({time.time()-t0:.0f}s)')
spy_df, _ = load_px(DATA_DIR/'SPY.csv'); rsp_df, _ = load_px(RSP_PATH)
SPY = dict(Date=spy_df['Date'].values, Close=spy_df['Close'].to_numpy(float))
RSP = dict(Date=rsp_df['Date'].values, Close=rsp_df['Close'].to_numpy(float))
assert max(v['Date'][-1] for v in price_db.values()) <= np.datetime64(CUT_DATE)
REG_PATH = CACHE/'regime.parquet'
if REG_PATH.exists():
    REGIME = pd.read_parquet(REG_PATH)
else:
    REGIME = regime_series(build_panels(PANEL_SRC, SPY['Date']), SPY, RSP, SPY['Date']); REGIME.to_parquet(REG_PATH)
_isd = REGIME[(REGIME.index >= IS_START) & (REGIME.index <= PURGE_DATE)]
REG_EDGES = {k: float(_isd[k].quantile(0.80)) for k in ['breadth_50', 'rsp_spy_20d']}   # IS trading days only
REGIME['on'] = (REGIME.breadth_50 > REG_EDGES['breadth_50']) & (REGIME.rsp_spy_20d > REG_EDGES['rsp_spy_20d'])
print(f'Universe {len(price_db)} | panel {len(PANEL_SRC)} | regime top-quintile edges (IS days): {REG_EDGES} '
      f'| gate-on share of IS days {REGIME.loc[_isd.index, "on"].mean():.3f} | {time.time()-t0:.0f}s')


In [ ]:
# ===== Cell 2 — Survivors: re-simulate their trades, assert = Stage 2 grid  (~1-3 min) =====
def _same(a, b):
    if isinstance(a, (bool, np.bool_)) or isinstance(b, (bool, np.bool_)): return bool(a) == bool(b)
    a, b = float(a), float(b)
    if np.isnan(a) and np.isnan(b): return True
    return abs(a - b) <= 1e-9 * max(1.0, abs(a))


@njit(cache=True)
def _sim_path(O, H, L, C, ATR, TR, e, n, fill, stop, target, trail, T, chand):
    """One trade, REALISTIC fills, bar order: stop (low) -> target (high) -> time (close of last bar) ->
    trailing signal at the close, exit at the NEXT open. Reasons: 0 stop 1 target 2 time 3 end-of-data 4 trail."""
    end = e + T; last = (end if end < n else n) - 1; hh = -1e300
    for i in range(e, last + 1):
        if L[i] <= stop:
            return i, (stop if i == e else min(O[i], stop)), 0
        if H[i] >= target:
            return i, (max(fill, target) if i == e else max(O[i], target)), 1
        if i == last:
            return i, C[i], (3 if end > n else 2)
        if trail == 4:
            if H[i] > hh: hh = H[i]
            if C[i] < hh - chand * ATR[i]: return i + 1, O[i + 1], 4
        elif trail > 0:
            if C[i] < TR[i]: return i + 1, O[i + 1], 4
    return last, C[last], 2

@njit(parallel=True, cache=True)
def run_grid(Of, Hf, Lf, Cf, ATRf, S10f, S21f, S50f, DAYf, off, nbar, nfull,
             f_tk, f_e, f_fill, f_edate, f_regime, f_month, f_year, f_in_is, STOPm, SKIPm, TGTm,
             c_stop, c_tgt, c_trail, c_time, chand, cost, n_months, n_years):
    nc = len(c_stop); nf = len(f_tk)
    N = np.zeros((nc, 2)); NWIN = np.zeros((nc, 2)); WR_ = np.zeros((nc, 2)); LR_ = np.zeros((nc, 2))
    WM = np.zeros((nc, 2, n_months)); LM = np.zeros((nc, 2, n_months)); CM = np.zeros((nc, 2, n_months))
    RY = np.zeros((nc, 2, n_years)); CY = np.zeros((nc, 2, n_years)); SRET = np.zeros((nc, 2)); SHELD = np.zeros((nc, 2))
    NCLIP = np.zeros((nc, 2))
    for c in prange(nc):
        s = c_stop[c]; g = c_tgt[c]; tr = c_trail[c]; T = c_time[c]
        for gate in range(2):
            last_tk = -1; last_exit = -(1 << 62); blocked = False
            for k in range(nf):
                tk = f_tk[k]
                if tk != last_tk:
                    last_tk = tk; last_exit = -(1 << 62); blocked = False
                if blocked: continue
                if gate == 1 and not f_regime[k]: continue
                if SKIPm[s, k]: continue
                if f_edate[k] <= last_exit: continue
                o = off[tk]; n = nbar[tk]; stop = STOPm[s, k]; fill = f_fill[k]
                TR = S10f[o:o+n] if tr == 1 else (S21f[o:o+n] if tr == 2 else S50f[o:o+n])
                xi, px, rc = _sim_path(Of[o:o+n], Hf[o:o+n], Lf[o:o+n], Cf[o:o+n], ATRf[o:o+n], TR,
                                       f_e[k], n, fill, stop, TGTm[g, k], tr, T, chand)
                if rc == 3 and nfull[tk] > n:          # time stop clipped by the cut: drop it and the ticker's later entries
                    blocked = True; NCLIP[c, gate] += 1; continue
                last_exit = DAYf[o + xi]
                if not f_in_is[k]: continue              # pre-2011 entries shape the overlap chain only
                risk = fill - stop; r = (px - fill) / risk; rc_ = r - cost * fill / risk
                ret = px / fill - 1.0 - cost
                N[c, gate] += 1; SRET[c, gate] += ret; SHELD[c, gate] += xi - f_e[k] + 1
                if r > 0: NWIN[c, gate] += 1; WR_[c, gate] += r
                else: LR_[c, gate] -= r
                m = f_month[k]; y = f_year[k]
                if rc_ > 0: WM[c, gate, m] += rc_
                else: LM[c, gate, m] -= rc_
                CM[c, gate, m] += 1; RY[c, gate, y] += ret; CY[c, gate, y] += 1
    return N, NWIN, WR_, LR_, WM, LM, CM, RY, CY, SRET, SHELD, NCLIP

EXIT_CFGS = list(itertools.product(range(len(STOPS)), range(len(TARGETS)), range(len(TRAILS)), TIMES))
TRAIL_CODE = {'none': 0, 'sma10': 1, 'sma21': 2, 'sma50': 3, 'chand3': 4}

def flat_prices(tickers):
    off, nb, nfl = [], [], []; cols = {k: [] for k in ['Open','High','Low','Close','ATR','SMA10','SMA21','SMA50']}; days = []; o = 0
    for tk in tickers:
        p = price_db[tk]; off.append(o); nb.append(len(p['Close'])); nfl.append(N_FULL[tk]); o += len(p['Close'])
        for k in cols: cols[k].append(p[k])
        days.append(p['Date'].astype('datetime64[D]').astype(np.int64))
    cat = {k: np.concatenate(v).astype(np.float64) for k, v in cols.items()}
    return cat, np.concatenate(days), np.array(off, np.int64), np.array(nb, np.int64), np.array(nfl, np.int64)

def fire_arrays(F, tk_index):
    """Fires sorted (ticker, entry_date, lch, rch) — the per-config non-overlap order."""
    F = F.sort_values(['ticker','entry_date','lch','rch'], kind='mergesort').reset_index(drop=True)
    fill = F.fill.to_numpy(float); atr = F.atr_dec.to_numpy(float)
    st = np.vstack([F.pitc_stop.to_numpy(float), fill*0.93, fill*0.92, fill - 2*atr, fill - 3*atr])
    skip = ~np.isfinite(st) | (st >= fill) | ((fill - st) / fill > MAX_STOP_PCT + 1e-12)
    rim, dep = F.rim.to_numpy(float), F.depth_abs.to_numpy(float)
    tg = np.vstack([rim+0.5*dep, rim+0.75*dep, rim+1.0*dep, rim+1.5*dep, fill*1.15, fill*1.20, fill*1.25, np.full(len(F), np.inf)])
    reg = REGIME['on'].reindex(pd.DatetimeIndex(F.dec_date)).fillna(False).to_numpy(bool)
    ed = F.entry_date
    in_is = ((ed >= IS_START) & (ed <= PURGE_DATE)).to_numpy()
    month = np.where(in_is, (ed.dt.year - IS_YEARS[0]) * 12 + ed.dt.month - 1, 0).astype(np.int64)
    year = np.where(in_is, ed.dt.year - IS_YEARS[0], 0).astype(np.int64)
    return dict(F=F, tk=F.ticker.map(tk_index).to_numpy(np.int64), e=F.e.to_numpy(np.int64), fill=fill,
                edate=ed.values.astype('datetime64[D]').astype(np.int64), regime=reg, month=month, year=year,
                in_is=in_is, STOP=st, SKIP=skip, TGT=tg)

def run_variant(vname, entry, P, A):
    cs = np.array([c[0] for c in EXIT_CFGS], np.int64); ct = np.array([c[1] for c in EXIT_CFGS], np.int64)
    ctr = np.array([TRAIL_CODE[TRAILS[c[2]]] for c in EXIT_CFGS], np.int64); cti = np.array([c[3] for c in EXIT_CFGS], np.int64)
    cat, days, off, nb, nfl = P
    return run_grid(cat['Open'], cat['High'], cat['Low'], cat['Close'], cat['ATR'], cat['SMA10'], cat['SMA21'], cat['SMA50'],
                    days, off, nb, nfl, A['tk'], A['e'], A['fill'], A['edate'], A['regime'], A['month'], A['year'], A['in_is'],
                    A['STOP'], A['SKIP'], A['TGT'], cs, ct, ctr, cti, CHAND_MULT, COST_PCT, N_MONTHS, len(IS_YEARS))

def sim_cutoff_test(A, P, n=200, seed=0, leak=False):
    """Exit causality: re-simulate sampled (fire, config) pairs on prices CUT at the exit bar, indicators
    re-derived from the cut data. Stop / target / time exits: bar xi fully known. Trailing exits (filled at
    bar xi's OPEN): bar xi's high/low/close masked. Exits by end-of-data are not sampled.
    leak=True: NEGATIVE CONTROL — an SMA trailing signal that peeks at the next close; must mismatch."""
    cat, days, off, nb, nfl = P; rng = np.random.default_rng(seed); bad = 0; tried = 0; guard = 0
    names = ['Open','High','Low','Close','ATR','SMA10','SMA21','SMA50']
    def sim(a, k, s, g, trn, T, nn):
        TR = a['SMA10'] if trn == 1 else (a['SMA21'] if trn == 2 else a['SMA50'])
        if leak:
            TR = TR.copy(); nxt = np.r_[a['Close'][1:], np.nan]
            TR = np.where(nxt < a['Close'], np.inf, TR)                     # peeks at C[i+1]
        return _sim_path(a['Open'], a['High'], a['Low'], a['Close'], a['ATR'], TR, A['e'][k], nn,
                         A['fill'][k], A['STOP'][s, k], A['TGT'][g, k], trn, T, CHAND_MULT)
    while tried < n and guard < 100 * n:
        guard += 1
        k = int(rng.integers(len(A['tk']))); s, g, t_, T = EXIT_CFGS[int(rng.integers(len(EXIT_CFGS)))]
        trn = TRAIL_CODE[TRAILS[t_]]
        if A['SKIP'][s, k] or (leak and trn not in (1, 2, 3)): continue
        tk = A['tk'][k]; o, n_ = off[tk], nb[tk]
        arr = {nm: cat[nm][o:o+n_].copy() for nm in names}
        xi, px, rc = sim(arr, k, s, g, trn, T, n_)
        if rc == 3 or (leak and rc != 4): continue
        tried += 1
        q = {nm: v[:xi+1].copy() for nm, v in arr.items()}
        if rc == 4:
            for nm in ('High', 'Low', 'Close'): q[nm][xi] = np.nan
        d = pd.DataFrame({'High': q['High'], 'Low': q['Low'], 'Close': q['Close']})
        q['ATR'] = compute_atr(d, ATR_PERIOD).to_numpy(float)
        for w in (10, 21, 50): q[f'SMA{w}'] = d['Close'].rolling(w).mean().to_numpy(float)
        xi2, px2, rc2 = sim(q, k, s, g, trn, T, xi + 1)
        if not (xi2 == xi and rc2 == rc and _same(px2, px)): bad += 1
    return bad

OUT_NAMES = ['N','NWIN','WR_','LR_','WM','LM','CM','RY','CY','SRET','SHELD','NCLIP']
GATE_COLS = ['gate1', 'gate2', 'gate3', 'gate4', 'gate5']

def trades_for_config(A, P, s, g, trn, T, gate):
    """Kept IS trades of one config — the run_grid loop, one trade per row."""
    cat, days, off, nb, nfl = P; rows = []; last_tk = -1; last_exit = -(1 << 62); blocked = False
    for k in range(len(A['tk'])):
        tk = int(A['tk'][k])
        if tk != last_tk: last_tk, last_exit, blocked = tk, -(1 << 62), False
        if blocked or (gate == 1 and not A['regime'][k]) or A['SKIP'][s, k] or A['edate'][k] <= last_exit: continue
        o, n = off[tk], nb[tk]; fill, stop, tgt = A['fill'][k], A['STOP'][s, k], A['TGT'][g, k]
        TR = cat['SMA10' if trn == 1 else ('SMA21' if trn == 2 else 'SMA50')][o:o+n]
        xi, px, rc = _sim_path(cat['Open'][o:o+n], cat['High'][o:o+n], cat['Low'][o:o+n], cat['Close'][o:o+n],
                               cat['ATR'][o:o+n], TR, A['e'][k], n, fill, stop, tgt, trn, T, CHAND_MULT)
        if rc == 3 and nfl[tk] > n: blocked = True; continue
        last_exit = days[o + xi]
        if not A['in_is'][k]: continue
        risk = fill - stop; R = (px - fill) / risk
        rows.append(dict(tk=tk, ticker=TK_LIST[tk], e=int(A['e'][k]), xi=int(xi), entry_day=int(A['edate'][k]),
                         exit_day=int(days[o + xi]), fill=fill, stop=stop, sd=risk / fill, px=px, rc=int(rc), R=R,
                         Rc=R - COST_PCT * fill / risk, ret=px / fill - 1 - COST_PCT, held=int(xi - A['e'][k] + 1),
                         month=int(A['month'][k]), year=int(A['year'][k])))
    return pd.DataFrame(rows)

def metrics(px, fill, stop, held):
    px, fill, stop, held = map(np.asarray, (px, fill, stop, held))
    if len(px) == 0: return dict(n=0, PF_cost=np.nan, avg_ret=np.nan, avg_bars=np.nan, ret_per_bar=np.nan)
    risk = fill - stop; rc = (px - fill) / risk - COST_PCT * fill / risk; ret = px / fill - 1 - COST_PCT
    w, l = rc[rc > 0].sum(), -rc[rc < 0].sum()
    return dict(n=len(px), PF_cost=(w / l if l > 0 else np.inf), avg_ret=ret.mean(), avg_bars=held.mean(),
                ret_per_bar=ret.sum() / held.sum())
# ---- run ----
CFG = pd.read_parquet(CACHE/'configs_all_gates.parquet')
SURV = CFG[CFG[GATE_COLS].all(axis=1)].sort_values('DSR', ascending=False).reset_index(drop=True)
print(f'Stage 2 survivors: {len(SURV)}'); assert len(SURV) > 0, 'no Stage 2 survivors — nothing to test'
print(SURV[['variant','entry','stop','target','trail','time','gate','n','PF_cost','DSR','DSR_rawN']].to_string(index=False))
for t in SURV.target:
    assert t in ('none', 'p15', 'p20', 'p25'), f'target {t} is defined by cup depth — no random-entry equivalent'
TK_LIST = sorted(price_db); TK_INDEX = {tk: i for i, tk in enumerate(TK_LIST)}; PRICES = flat_prices(TK_LIST)
TRADES = {}
for i, r in SURV.iterrows():
    F = pd.read_parquet(CACHE/f'fires_{r.variant}_{r.entry}.parquet'); A = fire_arrays(F[F.v2b_dec], TK_INDEX)
    s, g, t_ = STOPS.index(r.stop), TARGETS.index(r.target), TRAILS.index(r.trail); gi = GATES.index(r.gate)
    TR_ = trades_for_config(A, PRICES, s, g, TRAIL_CODE[r.trail], int(r.time), gi)
    Gz = np.load(CACHE/f'grid_{r.variant}_{r.entry}.npz'); c = EXIT_CFGS.index((s, g, t_, int(r.time)))
    G = {nm: Gz[f'arr_{j}'] for j, nm in enumerate(OUT_NAMES)}
    ok = (len(TR_) == G['N'][c, gi] and abs(TR_.Rc.clip(lower=0).sum() - G['WM'][c, gi].sum()) < 1e-8
          and abs((-TR_.Rc.clip(upper=0)).sum() - G['LM'][c, gi].sum()) < 1e-8 and abs(TR_.ret.sum() - G['SRET'][c, gi]) < 1e-8)
    print(f'survivor {i}: {len(TR_)} trades re-simulated | equals Stage 2 grid: {ok}')
    assert ok, 'survivor trades do not reproduce the Stage 2 grid — stop'
    TRADES[i] = TR_


In [ ]:
# ===== Cell 3 — SELECTION null (shared kernel, protocol-validated); TIMING DEFERRED  (~3-10 min) =====
TGT_MULT = {'none': np.inf, 'p15': 1.15, 'p20': 1.20, 'p25': 1.25}

def entry_candidates(p, leak=False):
    """Entry bars e = d+1 whose decision bar d has v2b (SMA50 > SMA200, closes <= d). leak: NEGATIVE CONTROL
    that requires the ENTRY bar's close to be up (peeks past the decision)."""
    v2b = p['S50'] > p['S200']
    d = np.flatnonzero(v2b[:-1]); e = d + 1
    if leak: e = e[p['Close'][e] > p['Close'][e - 1]]
    return e

def is_months(p, e):
    dt = pd.DatetimeIndex(p['Date'][e]); ok = (dt >= IS_START) & (dt <= PURGE_DATE)
    return e[ok], np.asarray((dt[ok].year - IS_YEARS[0]) * 12 + dt[ok].month - 1)

def build_pools(tickers):
    pm, ptm = {}, {}
    for tk in tickers:
        i = TK_INDEX[tk]; e, m = is_months(price_db[tk], entry_candidates(price_db[tk]))
        for mm in np.unique(m):
            ee = e[m == mm]; ptm[(i, int(mm))] = ee
            a, b = pm.setdefault(int(mm), ([], [])); a.append(np.full(len(ee), i)); b.append(ee)
    return {mm: (np.concatenate(a), np.concatenate(b)) for mm, (a, b) in pm.items()}, ptm

def sim_one(tk, e, fill, sd, cfg):
    cat, days, off, nb, nfl = PRICES; o, n = off[tk], nb[tk]; trn = TRAIL_CODE[cfg.trail]
    TR = cat['SMA10' if trn == 1 else ('SMA21' if trn == 2 else 'SMA50')][o:o+n]
    stop = fill * (1 - sd); tgt = fill * TGT_MULT[cfg.target]
    xi, px, rc = _sim_path(cat['Open'][o:o+n], cat['High'][o:o+n], cat['Low'][o:o+n], cat['Close'][o:o+n],
                           cat['ATR'][o:o+n], TR, int(e), n, fill, stop, tgt, trn, int(cfg.time), CHAND_MULT)
    clipped = rc == 3 and nfl[tk] > n
    return xi, px, stop, clipped, days[o + int(e)], days[o + xi]

def fixed_sd(cfg):
    return {'pct7': 0.07, 'pct8': 0.08}.get(cfg.stop)

def draw_set(TR_, cfg, pool_m, pool_tm, kind, rng, fill_of=None):
    """One month-matched draw. kind '1a' (survivor ticker set), '1b' (same ticker + month), '4b' (breakouts,
    pool carries fills). Non-overlap per ticker; rejected candidates are redrawn (60 tries, then shortfall)."""
    cat = PRICES[0]; off = PRICES[2]; fsd = fixed_sd(cfg); sd_m = TR_.groupby('month').sd.apply(np.array).to_dict()
    med_sd = float(TR_.sd.median()); iv = {}; px_, fl_, st_, hd_ = [], [], [], []; short = 0
    for r in TR_.sort_values('entry_day').itertuples():
        if kind == '1b':
            pe = pool_tm.get((r.tk, r.month)); ptk = None; pf = None
        else:
            got = pool_m.get(r.month); ptk, pe = (got[0], got[1]) if got else (None, None); pf = got[2] if (got and len(got) > 2) else None
        if pe is None or len(pe) == 0: short += 1; continue
        if fsd is not None: sd = fsd
        elif kind == '1b': sd = r.sd
        elif kind == '4b': sd = med_sd
        else: sd = float(rng.choice(sd_m[r.month]))
        for _ in range(60):
            j = int(rng.integers(len(pe))); tk = r.tk if kind == '1b' else int(ptk[j]); e = int(pe[j])
            if pf is not None:
                fill = float(pf[j])                                                  # 4b: the breakout's own fill
            elif cfg.entry == 'buy_stop':                                            # random BUY-STOP at the prior bar's high
                lvl = float(PRICES[0]['High'][off[tk] + e - 1])
                if not (PRICES[0]['High'][off[tk] + e] >= lvl): continue          # not hit -> redraw
                fill = max(float(cat['Open'][off[tk] + e]), lvl)
            else:
                fill = float(cat['Open'][off[tk] + e])
            xi, px, stop, clipped, a, b = sim_one(tk, e, fill, sd, cfg)
            if clipped: continue
            if any(not (a > ob or b < oa) for oa, ob in iv.get(tk, [])): continue
            iv.setdefault(tk, []).append((a, b)); px_.append(px); fl_.append(fill); st_.append(stop); hd_.append(xi - e + 1); break
        else:
            short += 1
    m = metrics(px_, fl_, st_, hd_); m['shortfall'] = short
    return m

def pct(v, dist):
    """Percentile of v in dist; values within 1e-9 relative count as TIES (same trades summed in another
    order differ by ~1e-15, which must not decide a percentile)."""
    d = np.asarray(dist, float); tie = np.abs(d - v) <= 1e-9 * max(1.0, abs(v))
    return 100.0 * (((d < v) & ~tie).mean() + 0.5 * tie.mean())

def tier_conditions(sv, null_1b, ew_mean, ew_ci_year_lo, draws_4b_52w, tier_pct=None):
    """The pre-registered conditions. sv: survivor metrics dict; null_1b / draws_4b_52w: DataFrames of draws."""
    tp = TIER_PCT if tier_pct is None else tier_pct
    c1b = pct(sv['PF_cost'], null_1b.PF_cost) >= tp and pct(sv['avg_ret'], null_1b.avg_ret) >= tp
    cew = bool(ew_mean > 0 and ew_ci_year_lo > 0)
    c4b = pct(sv['PF_cost'], draws_4b_52w.PF_cost) >= tp
    return bool(c1b), cew, bool(c4b)

def tier_of(c1b, cew, c4b):
    """TIER 1 = all three; TIER 2 = 1b and EW but not 4b; TIER 3 = fails 1b or EW."""
    return 1 if (c1b and cew and c4b) else (2 if (c1b and cew) else 3)

def cutoff_test_candidates(n=100, seed=0, leak=False):
    """v2b-at-decision entries must survive truncation at the entry bar with only its OPEN known."""
    rng = np.random.default_rng(seed); bad = 0; tried = 0
    while tried < n:
        tk = TK_LIST[int(rng.integers(len(TK_LIST)))]; p = price_db[tk]; e_all = entry_candidates(p, leak)
        if len(e_all) == 0: continue
        e = int(e_all[int(rng.integers(len(e_all)))]); tried += 1
        q = _trunc(p, e)
        for k in ('High', 'Low', 'Close', 'Volume'): q[k][e] = np.nan
        q = _rederive(q)
        bad += int(e not in set(entry_candidates(q, leak).tolist()))
    return bad

@njit(cache=True)
def _sim3(O, H, L, C, e, n, fill, stop, tgt, rim, mode, above0, skip_e_stop, T):
    """Stage 2 bar order + realistic fills; mode 2 = rim retest (after a close above the rim, the first close back
    below exits at the next open). skip_e_stop: stop not active on the fill bar (limit + handle stop)."""
    end = e + T; last = (end if end < n else n) - 1; above = above0
    for i in range(e, last + 1):
        if not (skip_e_stop and i == e):
            if L[i] <= stop: return i, (stop if i == e else min(O[i], stop)), 0
        if H[i] >= tgt: return i, (max(fill, tgt) if i == e else max(O[i], tgt)), 1
        if i == last: return i, C[i], (3 if end > n else 2)
        if mode == 2:
            if C[i] > rim: above = True
            elif above and C[i] < rim: return i + 1, O[i + 1], 4
    return last, C[last], 2


@njit(cache=True)
def null_draws2(Of, Hf, Lf, Cf, DAYf, off, nb, nfl, s_tk, s_lo, s_hi, p_tk, p_a, s_kind, s_sd, s_skip, s_tgt, s_rim,
                s_above, s_delta, s_live, mode, T, cost, n_draws, seed, n_tk, excl_own, purge_day):
    """Shared null kernel. Each slot draws a candidate (ticker, anchor bar a) from p_tk / p_a[s_lo:s_hi] and re-enters
    with the slot's order type from a: kind 0 market (fill = next open), 2 limit (C[a] x (1 - delta), live s_live bars),
    3 buy-stop (level H[a], fills on a+1 at max(open, level)). Matched stop distance / exit levels; non-overlap per
    ticker within a draw; 60 tries per slot, then shortfall. Per draw: n, sum Rc+, sum Rc-, sum ret, sum held, shortfall."""
    np.random.seed(seed); S = len(s_tk); res = np.zeros((n_draws, 6))
    ia = np.empty(S, np.int64); ib = np.empty(S, np.int64); nx = np.empty(S, np.int64); head = np.empty(n_tk, np.int64)
    for d in range(n_draws):
        head[:] = -1; cnt = 0
        for s in range(S):
            lo = s_lo[s]; hi = s_hi[s]; ok = False
            if hi > lo:
                for att in range(60):
                    j = lo + np.random.randint(hi - lo); tk = p_tk[j]; a = p_a[j]
                    if excl_own and tk == s_tk[s]: continue
                    o = off[tk]; n = nb[tk]; e = -1; fill = 0.0
                    if s_kind[s] == 2:
                        lim = Cf[o + a] * (1.0 - s_delta[s]); top = a + s_live[s]
                        if top > n - 1: top = n - 1
                        for b in range(a + 1, top + 1):
                            if Lf[o + b] <= lim:
                                e = b; break
                        if e < 0: continue
                        fill = min(Of[o + e], lim)
                    else:
                        e = a + 1
                        if e >= n: continue
                        if s_kind[s] == 3:
                            if not (Hf[o + e] >= Hf[o + a]): continue
                            fill = max(Of[o + e], Hf[o + a])
                        else:
                            fill = Of[o + e]
                    if DAYf[o + e] > purge_day: continue
                    stop = fill * (1.0 - s_sd[s])
                    tgt = fill * (1.0 + s_tgt[s]) if s_tgt[s] >= 0 else np.inf
                    xi, px, rc = _sim3(Of[o:o+n], Hf[o:o+n], Lf[o:o+n], Cf[o:o+n], e, n, fill, stop, tgt, fill * (1.0 + s_rim[s]),
                                       mode, s_above[s], s_skip[s], T)
                    if rc == 3 and nfl[tk] > n: continue
                    aa = DAYf[o + e]; bb = DAYf[o + xi]; q = head[tk]; clash = False
                    while q >= 0:
                        if not (aa > ib[q] or bb < ia[q]):
                            clash = True; break
                        q = nx[q]
                    if clash: continue
                    ia[cnt] = aa; ib[cnt] = bb; nx[cnt] = head[tk]; head[tk] = cnt; cnt += 1
                    risk = fill - stop; r = (px - fill) / risk - cost * fill / risk
                    res[d, 0] += 1
                    if r > 0: res[d, 1] += r
                    else: res[d, 2] -= r
                    res[d, 3] += px / fill - 1.0 - cost; res[d, 4] += xi - e + 1; ok = True
                    break
            if not ok: res[d, 5] += 1
    return res

@njit(cache=True)
def null_slot_means(Of, Hf, Lf, Cf, DAYf, off, nb, nfl, s_tk, s_lo, s_hi, p_tk, p_a, s_kind, s_sd, s_skip, s_tgt, s_rim,
                    s_above, s_delta, s_live, mode, T, cost, n_draws, seed, n_tk, excl_own, purge_day):
    """Same draws as null_draws2, accumulated PER SLOT over draws: (sum ret, sum cost-R, successes) — for paired tests."""
    np.random.seed(seed); S = len(s_tk); acc = np.zeros((S, 3))
    ia = np.empty(S, np.int64); ib = np.empty(S, np.int64); nx = np.empty(S, np.int64); head = np.empty(n_tk, np.int64)
    for d in range(n_draws):
        head[:] = -1; cnt = 0
        for s in range(S):
            lo = s_lo[s]; hi = s_hi[s]
            if hi <= lo: continue
            for att in range(60):
                j = lo + np.random.randint(hi - lo); tk = p_tk[j]; a = p_a[j]
                if excl_own and tk == s_tk[s]: continue
                o = off[tk]; n = nb[tk]; e = -1; fill = 0.0
                if s_kind[s] == 2:
                    lim = Cf[o + a] * (1.0 - s_delta[s]); top = a + s_live[s]
                    if top > n - 1: top = n - 1
                    for b in range(a + 1, top + 1):
                        if Lf[o + b] <= lim:
                            e = b; break
                    if e < 0: continue
                    fill = min(Of[o + e], lim)
                else:
                    e = a + 1
                    if e >= n: continue
                    if s_kind[s] == 3:
                        if not (Hf[o + e] >= Hf[o + a]): continue
                        fill = max(Of[o + e], Hf[o + a])
                    else:
                        fill = Of[o + e]
                if DAYf[o + e] > purge_day: continue
                stop = fill * (1.0 - s_sd[s])
                tgt = fill * (1.0 + s_tgt[s]) if s_tgt[s] >= 0 else np.inf
                xi, px, rc = _sim3(Of[o:o+n], Hf[o:o+n], Lf[o:o+n], Cf[o:o+n], e, n, fill, stop, tgt, fill * (1.0 + s_rim[s]),
                                   mode, s_above[s], s_skip[s], T)
                if rc == 3 and nfl[tk] > n: continue
                aa = DAYf[o + e]; bb = DAYf[o + xi]; q = head[tk]; clash = False
                while q >= 0:
                    if not (aa > ib[q] or bb < ia[q]):
                        clash = True; break
                    q = nx[q]
                if clash: continue
                ia[cnt] = aa; ib[cnt] = bb; nx[cnt] = head[tk]; head[tk] = cnt; cnt += 1
                risk = fill - stop
                acc[s, 0] += px / fill - 1.0 - cost; acc[s, 1] += (px - fill) / risk - cost * fill / risk; acc[s, 2] += 1
                break
    return acc

def block_q(x, blocks, reps, seed):
    """100 x share of block-bootstrap means > 0 (year blocks). q >= 95 <=> 90% CI lower bound > 0."""
    x = np.asarray(x, float); ub, inv = np.unique(blocks, return_inverse=True)
    S = np.bincount(inv, weights=x, minlength=len(ub)); Cn = np.bincount(inv, minlength=len(ub)).astype(float)
    idx = np.random.default_rng(seed).integers(0, len(ub), (reps, len(ub)))
    return 100.0 * float(((S[idx].sum(1) / Cn[idx].sum(1)) > 0).mean())

def paired_timing(T, slots_tim, tim, exit_, seed, n_draws=100, T_hold=None, reps=5000):
    """PAIRED timing test: per signal trade, its return minus the mean of its OWN timing draws (same ticker, market entry
    at a random bar in [t+1, t+21]); year-block bootstrap on the mean difference, for avg % return AND cost-R."""
    cat, days, off, nb, nfl = PRICES; lo, hi, ptk, pa = tim; sl = slots_tim
    acc = null_slot_means(cat['Open'], cat['High'], cat['Low'], cat['Close'], days, off, nb, nfl, sl['s_tk'], lo, hi, ptk, pa,
                          sl['s_kind'], sl['s_sd'], sl['s_skip'], sl['s_tgt'], sl['s_rim'], sl['s_above'], sl['s_delta'],
                          sl['s_live'], 2 if exit_ == 'X2b' else 0, T_hold or HOLD3, COST_PCT, n_draws, seed, len(TK_LIST), False, PURGE_DAY)
    ok = acc[:, 2] > 0
    d_ret = T.ret.to_numpy()[ok] - acc[ok, 0] / acc[ok, 2]; d_rc = T.Rc.to_numpy()[ok] - acc[ok, 1] / acc[ok, 2]; yrs = T.year.to_numpy()[ok]
    q_ret, q_rc = block_q(d_ret, yrs, reps, seed + 1), block_q(d_rc, yrs, reps, seed + 2)
    return dict(n_paired=int(ok.sum()), diff_ret=float(d_ret.mean()), diff_rc=float(d_rc.mean()), q_ret=q_ret, q_rc=q_rc,
                pass_paired_timing=bool(q_ret >= 95 and q_rc >= 95))

def leg_table(F, stop, exit_):
    """Every entry of F simulated on its OWN (no non-overlap): one leg per (ticker, left lip). Clipped legs are dropped
    (treated as missing). Returns ticker, lch, entry/exit day, net ret, initial stop distance, entry year."""
    cat, days, off, nb, nfl = PRICES; rows = []; mode = 2 if exit_ == 'X2b' else 0
    for f in F.to_dict('records'):
        st = stop_for(f, stop)
        if st is None or st >= f['fill']: continue
        tk = TK_INDEX[f['ticker']]; e = int(f['e']); o, n = off[tk], nb[tk]
        tgt = f['rim'] + 0.5 * f['depth_abs'] if exit_ == 'X2a' else np.inf
        xi, px, rc = _sim3(cat['Open'][o:o+n], cat['High'][o:o+n], cat['Low'][o:o+n], cat['Close'][o:o+n], e, n, f['fill'], st,
                           tgt, f['rim'], mode, bool(f['above0']), f['order'] == 'limit' and stop == 'handle', HOLD3)
        if rc == 3 and nfl[tk] > n: continue
        rows.append(dict(ticker=f['ticker'], lch=int(f['lch']), entry_day=int(days[o + e]), exit_day=int(days[o + xi]),
                         ret=px / f['fill'] - 1 - COST_PCT, sd=(f['fill'] - st) / f['fill'], entry_date=f['entry_date']))
    return pd.DataFrame(rows)

def paired_vs_breakout(pat, rule, stop, exit_, seed, reps=5000):
    """(B): per base (ticker, left lip), in-handle leg minus breakout leg, each as EW-excess (% and R); missing leg = 0;
    bases kept if the EARLIER leg enters inside the IS window; year-block bootstrap on the mean difference."""
    A = leg_table(ENTRIES3[(pat, rule)], stop, exit_); B = leg_table(ENTRIES3[(pat, 'bo_next')], stop, exit_)
    for X in (A, B):
        if len(X):
            X['ex_pct'] = [r.ret - ew_ret(r.entry_day, r.exit_day) for r in X.itertuples()]; X['ex_R'] = X.ex_pct / X.sd
    key = ['ticker', 'lch']; cols = key + ['entry_date', 'ex_pct', 'ex_R']
    M_ = (A[cols] if len(A) else pd.DataFrame(columns=cols)).merge(B[cols] if len(B) else pd.DataFrame(columns=cols), on=key, how='outer', suffixes=('_h', '_b'))
    M_['first'] = M_[['entry_date_h', 'entry_date_b']].min(axis=1)
    M_ = M_[(M_['first'] >= IS_START) & (M_['first'] <= PURGE_DATE)]
    M_ = M_[np.isfinite(M_.ex_pct_h.fillna(0)) & np.isfinite(M_.ex_pct_b.fillna(0))]
    if len(M_) < 20: return dict(pvb_n=int(len(M_)), pass_pvb=False, pvb_note='fewer than 20 bases')
    d_pct = M_.ex_pct_h.fillna(0) - M_.ex_pct_b.fillna(0); d_R = M_.ex_R_h.fillna(0) - M_.ex_R_b.fillna(0)
    yrs = pd.DatetimeIndex(M_['first']).year.to_numpy()
    q_pct, q_R = block_q(d_pct.to_numpy(), yrs, reps, seed), block_q(d_R.to_numpy(), yrs, reps, seed + 1)
    return dict(pvb_n=int(len(M_)), pvb_both=int((M_.ex_pct_h.notna() & M_.ex_pct_b.notna()).sum()),
                pvb_only_handle=int((M_.ex_pct_h.notna() & M_.ex_pct_b.isna()).sum()), pvb_only_bo=int((M_.ex_pct_h.isna() & M_.ex_pct_b.notna()).sum()),
                pvb_diff_pct=float(d_pct.mean()), pvb_diff_R=float(d_R.mean()), pvb_q_pct=q_pct, pvb_q_R=q_R,
                pass_pvb=bool(q_pct >= 95 and q_R >= 95))

def selection_pool(kind):
    """Per calendar date: (ticker, bar) with v2b true at that bar (turn: also C[b] > H[b-1]) and a next bar to enter."""
    cal = np.asarray(SPY['Date']).astype('datetime64[D]').astype(np.int64); parts = []
    for tk, i in TK_INDEX.items():
        p = price_db[tk]; n = len(p['Close']); v2b = p['S50'] > p['S200']; b = np.flatnonzero(v2b[:n - 1])
        if kind == 'turn': b = b[(b >= 1) & (p['Close'][b] > p['High'][b - 1])]
        dd = p['Date'][b].astype('datetime64[D]').astype(np.int64); cp = np.searchsorted(cal, dd)
        ok = (cp < len(cal)) & (cal[np.minimum(cp, len(cal) - 1)] == dd)
        parts.append((cp[ok], np.full(int(ok.sum()), i, np.int64), b[ok]))
    cp = np.concatenate([x[0] for x in parts]); tk = np.concatenate([x[1] for x in parts]); bb = np.concatenate([x[2] for x in parts])
    order = np.argsort(cp, kind='mergesort'); cp, tk, bb = cp[order], tk[order], bb[order]
    ptr = np.searchsorted(cp, np.arange(len(cal) + 1))
    return dict(cal=cal, ptr=ptr, tk=tk.astype(np.int64), bar=bb.astype(np.int64))

def kind_code(order): return 2 if order == 'limit' else (3 if order == 'buystop' else 0)

def null_inputs(T, rule):
    """Slot arrays + TIMING (own ticker, bars t+1..t+21, unconditioned) and SELECTION (same date, other tickers, v2b,
    turn trigger for turn) candidate pools for one config's trades (T sorted by entry_day)."""
    cat, days, off, nb, nfl = PRICES; W = GATE3['timing_window']
    tl, th, ttk, ta = [], [], [], []; k = 0
    for r in T.itertuples():
        p = price_db[r.ticker]; n = len(p['Close']); a = np.arange(r.anchor + 1, min(r.anchor + 1 + W, n - 1))
        tl.append(k); th.append(k + len(a)); ttk.append(np.full(len(a), r.tk, np.int64)); ta.append(a.astype(np.int64)); k += len(a)
    SP = SEL_POOLS['turn' if rule == 'turn' else 'any']; L_ = len(SP['cal'])
    adays = np.array([days[off[r.tk] + r.anchor] for r in T.itertuples()], np.int64)
    cp = np.minimum(np.searchsorted(SP['cal'], adays), L_ - 1); okd = SP['cal'][cp] == adays
    slo = np.where(okd, SP['ptr'][cp], 0); shi = np.where(okd, SP['ptr'][cp + 1], 0)
    slots = dict(s_tk=T.tk.to_numpy(np.int64), s_kind=np.array([kind_code(o) for o in T.order], np.int64),
                 s_sd=T.sd.to_numpy(float), s_skip=T.skip.to_numpy(bool), s_tgt=T.tgt_dist.to_numpy(float),
                 s_rim=T.rim_dist.to_numpy(float), s_above=T.above0.to_numpy(bool), s_delta=T.delta0.to_numpy(float),
                 s_live=T.live.to_numpy(np.int64))
    timing = (np.array(tl, np.int64), np.array(th, np.int64), (np.concatenate(ttk) if ttk else np.zeros(0, np.int64)),
              (np.concatenate(ta) if ta else np.zeros(0, np.int64)))
    selection = (slo.astype(np.int64), shi.astype(np.int64), SP['tk'], SP['bar'])
    return slots, timing, selection

def run_null(slots, pools, exit_, excl_own, seed, n_draws=None, T=None):
    cat, days, off, nb, nfl = PRICES; lo, hi, ptk, pa = pools
    return null_draws2(cat['Open'], cat['High'], cat['Low'], cat['Close'], days, off, nb, nfl, slots['s_tk'], lo, hi, ptk, pa,
                       slots['s_kind'], slots['s_sd'], slots['s_skip'], slots['s_tgt'], slots['s_rim'], slots['s_above'],
                       slots['s_delta'], slots['s_live'], 2 if exit_ == 'X2b' else 0, T or HOLD3, COST_PCT,
                       n_draws or GATE3['n_draws'], seed, len(TK_LIST), excl_own, PURGE_DAY)

def pct(v, dist):
    d = np.asarray(dist, float); tie = np.abs(d - v) <= 1e-9 * max(1.0, abs(v))
    return 100.0 * (((d < v) & ~tie).mean() + 0.5 * tie.mean())

def draw_metrics(D):
    pf_ = np.where(D[:, 2] > 0, D[:, 1] / np.where(D[:, 2] > 0, D[:, 2], 1), np.inf); ret_ = D[:, 3] / np.maximum(D[:, 0], 1)
    return pf_, ret_

def block_boot(x, blocks, reps, seed, level):
    x = np.asarray(x, float); ub, inv = np.unique(blocks, return_inverse=True)
    S = np.bincount(inv, weights=x, minlength=len(ub)); Cn = np.bincount(inv, minlength=len(ub)).astype(float)
    idx = np.random.default_rng(seed).integers(0, len(ub), (reps, len(ub))); mb = S[idx].sum(1) / Cn[idx].sum(1); a = (1 - level) / 2
    return float(np.quantile(mb, a)), float(np.quantile(mb, 1 - a))

def survivor_slots(TR_, cfg):
    """Stage 2b survivor trades -> the columns the shared kernel needs."""
    assert cfg.trail == 'none', 'trailing exits are not supported by the shared null kernel'
    TGT = {'none': -1.0, 'p15': 0.15, 'p20': 0.20, 'p25': 0.25}
    return TR_.assign(anchor=TR_.e - 1, order='market' if cfg.entry == 'next_open' else 'buystop',
                      tgt_dist=TGT[cfg.target], rim_dist=0.0, above0=False, skip=False, delta0=0.0, live=0)
# ---- run ----
_trunc = lambda p, last: {k: (v[:last+1].copy() if isinstance(v, np.ndarray) else v) for k, v in p.items()}
def _rederive(q):
    d = pd.DataFrame({'High': q['High'], 'Low': q['Low'], 'Close': q['Close']})
    q['ATR'] = compute_atr(d, ATR_PERIOD).to_numpy(float)
    for k, w in (('S50', STOCK_FAST_SMA), ('S200', STOCK_SLOW_SMA), ('SMA10', 10), ('SMA21', 21), ('SMA50', 50)):
        q[k] = d['Close'].rolling(w).mean().to_numpy(float)
    return q
PURGE_DAY = int(np.datetime64(PURGE_DATE, 'D').astype(np.int64))
SEL_POOLS = {k: selection_pool(k) for k in ('any', 'turn')}
NULLS = {}
for i, cfg in SURV.iterrows():
    T = survivor_slots(TRADES[i], cfg).sort_values('entry_day').reset_index(drop=True); t0 = time.time()
    slots, _tim_unused, sel = null_inputs(T, 'bo_next' if cfg.entry == 'next_open' else 'bo_stop')
    for nm, pools, excl, sd_ in (('sel', sel, True, 8000 + i),):
        Dn = run_null(slots, pools, 'X1', excl, sd_, n_draws=N_DRAWS, T=int(cfg.time)); dpf, dret = draw_metrics(Dn)
        NULLS[(i, nm)] = pd.DataFrame(dict(n=Dn[:, 0], PF_cost=dpf, avg_ret=dret, avg_bars=Dn[:, 4] / np.maximum(Dn[:, 0], 1),
                                           ret_per_bar=Dn[:, 3] / np.maximum(Dn[:, 4], 1), shortfall=Dn[:, 5]))
    print(f'survivor {i}: SELECTION x {N_DRAWS} draws ({time.time()-t0:.0f}s) | mean shortfall {NULLS[(i, "sel")].shortfall.mean():.1f} of {len(T)} slots')


In [ ]:
# ===== Cell 4 — FIXED-HORIZON excess vs the EW universe and SPY (SUPPORTING only)  (~1-3 min) =====
HORIZONS = [60, 120]   # 20 and 250 dropped on synthetic-validity grounds (see header), before any real-data run

def fixed_horizon_excess(TR_):
    """Per trade and horizon h: the stock's own return from the CLOSE of the entry (fill) bar to the close h bars later,
    and the EW universe / SPY over the SAME close-to-close window. Starts AFTER the fill bar, so the intrabar move that
    triggered a buy-stop or limit fill is excluded (an open-start window was conditioned on it: buy-stop q 99, limit q 7
    on pure noise), and is independent of the trade's stop / exit (v1's EW row used each trade's own exit -> biased)."""
    out = {h_: [] for h_ in HORIZONS}
    for t in TR_.itertuples():
        p = price_db[t.ticker]; n = len(p['Close']); dts = p['Date']; e = int(t.e)
        ie = POS.get(int(np.datetime64(dts[e], 'D').astype(np.int64)))
        for h_ in HORIZONS:
            x = e + h_
            ix = POS.get(int(np.datetime64(dts[x], 'D').astype(np.int64))) if x < n else None
            if ie is None or ix is None: out[h_].append((np.nan, np.nan, np.nan)); continue
            r = PC[ix] / PC[ie] - 1
            out[h_].append((p['Close'][x] / p['Close'][e] - 1, float(np.nanmean(r)) if np.isfinite(r).any() else np.nan, SPYC[ix] / SPYC[ie] - 1))
    return {h_: np.array(v, float).reshape(-1, 3) for h_, v in out.items()}

def year_ci(x, blocks, reps, seed, level):
    """Year-block bootstrap of the mean: (CI lo, CI hi, q = 100 x share of bootstrap means > 0)."""
    x = np.asarray(x, float); ub, inv = np.unique(blocks, return_inverse=True)
    S = np.bincount(inv, weights=x, minlength=len(ub)); Cn = np.bincount(inv, minlength=len(ub)).astype(float)
    idx = np.random.default_rng(seed).integers(0, len(ub), (reps, len(ub))); mb = S[idx].sum(1) / Cn[idx].sum(1); a = (1 - level) / 2
    return float(np.quantile(mb, a)), float(np.quantile(mb, 1 - a)), 100.0 * float((mb > 0).mean())

def fh_table(TR_, seed):
    FH = fixed_horizon_excess(TR_); yrs = TR_.year.to_numpy(); out = {}
    for h_ in HORIZONS:
        X = FH[h_]; ok = np.isfinite(X).all(1)
        for bm, col in (('ew', 1), ('spy', 2)):
            d = X[ok, 0] - X[ok, col]
            lo, hi, q = year_ci(d, yrs[ok], BOOT_REPS, seed + h_ + col, CI_LEVEL) if ok.sum() >= 20 else (np.nan, np.nan, np.nan)
            out[f'{bm}_{h_}'] = dict(n=int(ok.sum()), mean=float(d.mean()) if ok.any() else np.nan, ci_lo=lo, ci_hi=hi, q=q)
    return out
# ---- run ----
CALD = pd.DatetimeIndex(SPY['Date']); POS = {d: i for i, d in enumerate(CALD.values.astype('datetime64[D]').astype(np.int64))}
PO = pd.DataFrame({tk: pd.Series(price_db[tk]['Open'], index=pd.DatetimeIndex(price_db[tk]['Date'])) for tk in TK_LIST}).reindex(CALD).to_numpy(float)
PC = pd.DataFrame({tk: pd.Series(price_db[tk]['Close'], index=pd.DatetimeIndex(price_db[tk]['Date'])) for tk in TK_LIST}).reindex(CALD).to_numpy(float)
SPYO = spy_df.set_index('Date')['Open'].reindex(CALD).to_numpy(float); SPYC = SPY['Close']
EXCESS = {}
for i, TR_ in TRADES.items():
    EXCESS[i] = fh_table(TR_, 700 + 10 * i)
    print(f'survivor {i} fixed-horizon excess vs EW: ' + ' | '.join(
        f"{h_}b {EXCESS[i][f'ew_{h_}']['mean']:+.4f} [{EXCESS[i][f'ew_{h_}']['ci_lo']:+.4f}, {EXCESS[i][f'ew_{h_}']['ci_hi']:+.4f}]" for h_ in HORIZONS))


In [ ]:
# ===== Cell 5 — Capital efficiency: return per bar held  (~seconds) =====
# Benchmarks over each trade's OWN window were dropped (same stop-window bias as v1's EW row); IS buy-and-hold kept.
# ---- run ----
i0 = POS[np.datetime64(CALD[CALD >= IS_START][0], 'D').astype(np.int64)]; i1 = POS[np.datetime64(CALD[CALD <= PURGE_DATE][-1], 'D').astype(np.int64)]
bars_is = i1 - i0 + 1
spy_bh = (SPYC[i1] / SPYO[i0] - 1) / bars_is
_r = PC[i1] / PO[i0] - 1; ew_bh = float(np.nanmean(_r)) / bars_is
EFF = {}
for i, TR_ in TRADES.items():
    EFF[i] = dict(survivor=TR_.ret.sum() / TR_.held.sum(), spy_buy_hold_IS=spy_bh, ew_buy_hold_IS=ew_bh)
    print(f'survivor {i} return/bar: ' + ' | '.join(f'{k} {v*1e4:.2f}bp ({v*252:.1%}/yr)' for k, v in EFF[i].items()))


In [ ]:
# ===== Cell 6 — Generic breakout null: 4a full strategy, 4b month-matched draws  (~2-5 min) =====
def bo_signal(p, N, entry_type, leak=False):
    """Breakout over the prior N-bar high (today excluded), v2b at the decision bar.
    next_open: first CLOSE above the level after a close at/below it; fill next open.
    buy_stop : armed at the close of d-1 (v2b, bar d-1 did not reach its level); fills on bar d when
               High >= level at max(Open, level).  leak: NEGATIVE CONTROL peeking at the entry bar's close."""
    H, C, O = p['High'], p['Close'], p['Open']; n = len(C)
    lvl = pd.Series(H).shift(1).rolling(N, min_periods=N).max().to_numpy()
    v2b = p['S50'] > p['S200']
    if entry_type == 'next_open':
        prev = np.r_[False, C[:-1] <= lvl[:-1]]
        d = np.flatnonzero((C > lvl) & prev & v2b); d = d[d + 1 < n]; e = d + 1; fill = O[e]
    else:
        prev = np.r_[False, (H[:-1] < lvl[:-1]) & v2b[:-1]]
        d = np.flatnonzero(prev & (H >= lvl)); e = d; fill = np.maximum(O[d], lvl[d])
    if leak:
        k = C[e] > fill; e, fill = e[k], fill[k]
    return e, fill

def bo_pool(N, entry_type):
    pm = {}
    for tk, i in TK_INDEX.items():
        p = price_db[tk]; e, fill = bo_signal(p, N, entry_type)
        if len(e) == 0: continue
        dt = pd.DatetimeIndex(p['Date'][e]); ok = (dt >= IS_START) & (dt <= PURGE_DATE)
        m = (dt[ok].year - IS_YEARS[0]) * 12 + dt[ok].month - 1
        for mm, ee, ff in zip(m, e[ok], fill[ok]):
            pm.setdefault(int(mm), ([], [], []))
            pm[int(mm)][0].append(i); pm[int(mm)][1].append(int(ee)); pm[int(mm)][2].append(float(ff))
    return {mm: (np.array(a), np.array(b), np.array(c)) for mm, (a, b, c) in pm.items()}

def bo_full(N, cfg, sd):
    """4a: every breakout signal on every ticker, per-ticker non-overlap, IS entries only."""
    cat, days, off, nb, nfl = PRICES; px_, fl_, st_, hd_ = [], [], [], []
    for tk, i in TK_INDEX.items():
        p = price_db[tk]; e, fill = bo_signal(p, N, cfg.entry); last = -(1 << 62)
        for ee, ff in zip(e, fill):
            day = days[off[i] + ee]
            if day <= last: continue
            xi, px, stop, clipped, a, b = sim_one(i, ee, ff, sd, cfg)
            if clipped: break
            last = b
            dt = pd.Timestamp(p['Date'][ee])
            if IS_START <= dt <= PURGE_DATE:
                px_.append(px); fl_.append(ff); st_.append(stop); hd_.append(xi - ee + 1)
    return metrics(px_, fl_, st_, hd_)

def cutoff_test_bo(N, entry_type, n=100, seed=0, leak=False):
    rng = np.random.default_rng(seed); bad = 0; tried = 0
    while tried < n:
        tk = TK_LIST[int(rng.integers(len(TK_LIST)))]; p = price_db[tk]; e_all, f_all = bo_signal(p, N, entry_type, leak)
        if len(e_all) == 0: continue
        j = int(rng.integers(len(e_all))); e = int(e_all[j]); tried += 1
        q = _trunc(p, e)
        for k in (('High', 'Low', 'Close', 'Volume') if entry_type == 'next_open' else ('Low', 'Close', 'Volume')): q[k][e] = np.nan
        q = _rederive(q); e2, f2 = bo_signal(q, N, entry_type, leak)
        hit = np.flatnonzero(e2 == e)
        bad += int(len(hit) != 1 or not _same(f2[hit[0]], f_all[j]))
    return bad
# ---- run ----
BO = {}
for i, cfg in SURV.iterrows():
    TR_ = TRADES[i]; sd = fixed_sd(cfg) or float(TR_.sd.median())
    for N in BREAKOUT_LOOKBACKS:
        if (N, cfg.entry) not in BO:
            ctrl = cutoff_test_bo(N, cfg.entry, 30, 1, leak=True); bad = cutoff_test_bo(N, cfg.entry, 100, 0)
            print(f'breakout {N}-bar {cfg.entry} cut-off test: 100 signals, {bad} mismatches | negative control {ctrl}/30 caught')
            assert ctrl > 0 and bad == 0
            BO[(N, cfg.entry)] = bo_pool(N, cfg.entry)
        t0 = time.time(); rng = np.random.default_rng(2000 + i + N)
        BO[(i, N, 'full')] = bo_full(N, cfg, sd)
        BO[(i, N, 'draws')] = pd.DataFrame([draw_set(TR_, cfg, BO[(N, cfg.entry)], None, '4b', rng) for _ in range(N_DRAWS)])
        print(f'survivor {i} breakout {N}: full n={BO[(i, N, "full")]["n"]} | {N_DRAWS} draws ({time.time()-t0:.0f}s)')


In [ ]:
# ===== Cell 7 — Side-by-side + TIERS (selection-only run)  (~seconds) =====
def null_pass(sv, D, tp=None):
    tp = TIER_PCT if tp is None else tp
    return bool(pct(sv['PF_cost'], D.PF_cost) >= tp and pct(sv['avg_ret'], D.avg_ret) >= tp)

def tier_sel(c_sel):
    """TIER S = selection passes (timing untested) · TIER 3 = selection fails."""
    return 'S' if c_sel else '3'
# ---- run ----
print('READING RULE (selection-only run): TIER S = SELECTION >= 95 on PF_cost AND avg_ret (timing untested) | TIER 3 = selection fails. '
      '4b and EW excess are supporting only. v1 tiers SUPERSEDED. Lockbox sealed.')
M = ['n', 'PF_cost', 'avg_ret', 'avg_bars', 'ret_per_bar']
NAMES = {'S': 'selection passes (timing untested)', '3': 'no detectable alpha (selection fails)'}
rows, tiers = [], []
fmt_d = lambda D, k: f'{D[k].median():.4g} [{D[k].quantile(.05):.4g}, {D[k].quantile(.95):.4g}]'
for i, cfg in SURV.iterrows():
    TR_ = TRADES[i]; sv = metrics(TR_.px, TR_.fill, TR_.stop, TR_.held); ex = EXCESS[i]
    label = f'{cfg.variant} {cfg.entry} {cfg.stop} {cfg.target} {cfg.trail} {cfg.time} {cfg.gate}'
    R = [dict(row='SURVIVOR', **{k: f'{sv[k]:.4g}' for k in M}, pct_PF='', pct_ret='')]
    for nm, D in (('SELECTION null (same day, other stock)', NULLS[(i, 'sel')]),
                  ('4b 52w breakout draws [supporting]', BO[(i, 252, 'draws')]), ('4b 50d breakout draws [supporting]', BO[(i, 50, 'draws')])):
        R.append(dict(row=nm, **{k: fmt_d(D, k) for k in M}, pct_PF=f'{pct(sv["PF_cost"], D.PF_cost):.1f}', pct_ret=f'{pct(sv["avg_ret"], D.avg_ret):.1f}'))
    for N in BREAKOUT_LOOKBACKS:
        f = BO[(i, N, 'full')]; R.append(dict(row=f'4a {N}-bar breakout (full) [supporting]', **{k: f'{f[k]:.4g}' for k in M}, pct_PF='', pct_ret=''))
    for k in ('spy_buy_hold_IS', 'ew_buy_hold_IS'):
        R.append(dict(row=k, n='', PF_cost='', avg_ret='', avg_bars='', ret_per_bar=f'{EFF[i][k]:.4g}', pct_PF='', pct_ret=''))
    c_sel = null_pass(sv, NULLS[(i, 'sel')]); c_4b = pct(sv['PF_cost'], BO[(i, 252, 'draws')].PF_cost) >= TIER_PCT
    t = tier_sel(c_sel)
    print(f'\n=================== SURVIVOR {i}: {label} | DSR effective-N {cfg.DSR:.3f} | raw-N {cfg.DSR_rawN:.3f}')
    print(pd.DataFrame(R).to_string(index=False))
    print('fixed-horizon excess [supporting] (stock open->close h bars later minus benchmark; year-block 90% CI):')
    print(pd.DataFrame([dict(horizon=h_, **{f'{bm}_{k}': ex[f'{bm}_{h_}'][k] for bm in ('ew', 'spy') for k in ('n', 'mean', 'ci_lo', 'ci_hi')})
                        for h_ in HORIZONS]).to_string(index=False, float_format=lambda v: f'{v:+.4f}'))
    print(f'>>> TIER {t} — {NAMES[t]}  (SELECTION: {c_sel} | supporting: 4b 52w {c_4b}, fixed-horizon EW CI lo > 0 at {"/".join(map(str, HORIZONS))}b: '
          f'{[ex[f"ew_{h_}"]["ci_lo"] > 0 for h_ in HORIZONS]})')
    tiers.append(dict(survivor=i, config=label, config_id=cfg.config_id, tier=t, pass_selection=c_sel, timing='DEFERRED',
                      pass_4b_supporting=c_4b, **{f'ew_fh{h_}_mean': ex[f'ew_{h_}']['mean'] for h_ in HORIZONS},
                      **{f'ew_fh{h_}_ci_lo': ex[f'ew_{h_}']['ci_lo'] for h_ in HORIZONS}, DSR=cfg.DSR, DSR_rawN=cfg.DSR_rawN))
    for r in R: rows.append(dict(survivor=i, config=label, **r))
pd.DataFrame(rows).to_csv(OUT2B/'controls_stage2b_v2.csv', index=False)
TIERS = pd.DataFrame(tiers); TIERS.to_csv(OUT2B/'tiers_stage2b_v2.csv', index=False)
if OUT2B_V1.exists():
    (OUT2B_V1/'SUPERSEDED.txt').write_text('Stage 2b v1 tiers are VOID: null 1b (same ticker, same month) is path-biased. '
        "v1's EW-excess row is ALSO BIASED UPWARD: its benchmark ran over each trade's own window, which ends at the trade's stop "
        '(stops fire mostly when the shared market falls). Use the fixed-horizon excess in v2. '
        f'See {OUT2B.name}/tiers_stage2b_v2.csv (selection-only run, timing deferred; written {pd.Timestamp.now(tz="UTC").isoformat()}).\n')
print('\n', TIERS.to_string(index=False)); print('\nsaved to', OUT2B, '| v1 marked SUPERSEDED | TIMING DEFERRED | LOCKBOX SEALED')
